In [1]:
from datetime import date
import hisepy
import pandas as pd
import polars as pl
import re
import os

In [2]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

### Lab grouping

In [4]:
grouping_uuid = '0180d34e-bcb4-4b81-9214-818641ec54e3'
grouping_csv = hisepy.cache_files([grouping_uuid])[0]
grouping = pl.read_csv(grouping_csv)

In [5]:
grouping.head()

category_name,lab,column_name
str,str,str
"""Anthropometric measures""","""Body Mass Index (BMI)""","""am.bmi"""
"""Anthropometric measures""","""Height""","""am.height"""
"""Anthropometric measures""","""Weight""","""am.weight"""
"""Blood Chemistry""","""Alanine Transaminase (ALT)""","""chem.alt"""
"""Blood Chemistry""","""Albumin""","""chem.albumin"""


In [6]:
lab_names = dict(zip(grouping['lab'], grouping['column_name']))

### Sample metadata

In [7]:
meta_uuid = '2da66a1a-17cc-498b-9129-6858cf639caf'

In [8]:
meta_file = hisepy.cache_files([meta_uuid])[0]

In [9]:
sample_meta = pd.read_csv(meta_file)

Convert drawDate to drawYear

In [10]:
sample_meta['sample.drawYear'] = [re.sub('-.+', '', d) for d in sample_meta['sample.drawDate']]
sample_meta = sample_meta.drop(['sample.drawDate'], axis = 1)

In [11]:
sample_meta['sample.drawYear'] = sample_meta['sample.drawYear'].astype(int)

Add Age Group, Age at First Draw, and Age at Draw

In [12]:
age_groups = {
    'UP1': 'Children',
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}
sample_meta['subject.ageGroup'] = [age_groups[cohort] for cohort in sample_meta['cohort.cohortGuid']]

In [13]:
sample_meta['subject.ageAtFirstDraw'] = sample_meta['sample.drawYear'] - sample_meta['subject.birthYear']
sample_meta['sample.subjectAgeAtDraw'] = sample_meta['sample.drawYear'] - sample_meta['subject.birthYear']

#### Standardize column names

In [14]:
sample_meta = pl.DataFrame(sample_meta)

In [15]:
meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'sample.visitName',
    'sample.visitDetails',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit',
    'sample.diseaseStatesRecordedAtVisit'
]

In [16]:
meta = sample_meta.select(meta_cols)

In [17]:
meta.shape

(108, 15)

## Query HISE to get labs

First, we make a dictionary (with curly braces) that defines what we want to get. In this case, we want to get samples from our study that were previously specified in our sample metadata.

Each entry in the dictionary has to be a list (square braces), even if it has a single entry.

In [18]:
query_dict = {
    'sampleKitGuid': meta['sample.sampleKitGuid'].to_list()
}

Now, we send this dictionary to HISE via hisepy

In [19]:
sample_data = hisepy.reader.read_samples(
    query_dict = query_dict,
    to_df = False
)

### Assemble labs from HISE data
This make a dictionary for each sampleKitGuid, and looks for non-null values in all of the results from HISE for that kit.

This is because there are multiple entries for each kit spread across HISE projects, and we're not certain that all labs are stored in all projects. This provides a way to assemble results that are stored in any project.

In [20]:
sample_data[0].keys()

dict_keys(['id', 'lastUpdated', 'latestBatchUpdate', 'labLastModified', 'surveyLastModified', 'projectGuid', 'subject', 'sample', 'specimens', 'hasLabResults', 'lab', 'survey', 'experiments', 'experimentComments', 'batchIdList', 'panelIdList', 'batchIds', 'panelIds'])

In [21]:
sample_data[0]['labLastModified']

'2022-10-07T18:21:02.71Z'

In [22]:
null_values = [None, '', 'NA', 999, 999000, 'None', 'nan', 'NaN']

In [23]:
lab_list = {}
mod_list = {}
i = 0
for data in sample_data:
    lab_data = {}
    kit = data['sample']['sampleKitGuid']
    lm = data['labLastModified']
    if not kit in lab_list.keys():
        lab_list[kit] = {}
        for lab in grouping['lab'].to_list():
            lab_list[kit][lab] = None
        mod_list[kit] = lab_list[kit].copy()
    
    for lab in grouping['lab'].to_list():
        if lab in data['lab']['labResults'].keys():
            value = data['lab']['labResults'][lab]
            # Check for missing values, set to None
            if value in null_values:
                value = None
            # Check for out of range values that will cause type errors. We'll drop the < or > and keep the value.
            elif isinstance(value, str):
                if '>' in value:
                    value = re.sub('>','',value)
                    value = float(value)
                elif '<' in value:
                    value = re.sub('<','',value)
                    value = float(value)
                    
            # If we still have a value, check for conflicts with the previous value
            if not value is None:
                if not lab_list[kit][lab] is None:
                    if lab_list[kit][lab] != value:
                        old_value = lab_list[kit][lab]
                        new_value = value
                        old_lm = mod_list[kit][lab]
                        new_lm = lm
                        print(f'Conflict: {kit} {lab} | ({old_lm}) {old_value} != {new_value} ({new_lm})')
                        # If there is a conflict, figure out which is newer
                        if old_lm > new_lm:
                            value = old_value
                            lm = old_lm
                            print(f'Keeping newer {old_value} ({old_lm})')
                        else:
                            print(f'Keeping newer {new_value} ({new_lm})')
                        
                lab_list[kit][lab] = value
                mod_list[kit][lab] = lm

In [24]:
lab_df = pd.DataFrame(lab_list).transpose()

In [25]:
lab_df = lab_df.reset_index(drop = False, names = 'sample.sampleKitGuid')

In [26]:
lab_df.shape

(108, 60)

Replace a SED Rate value that's misplaced

In [27]:
lab_df['SED Rate-Westergren (ESR)'] = lab_df['SED Rate-Westergren (ESR)'].replace('Yes',None)

In [28]:
all_bmi = []
for i in range(lab_df.shape[0]):
    wt = lab_df['Weight'].loc[i]
    ht = lab_df['Height'].loc[i]

    if ht is not None:
        if wt is not None:
            wt = float(wt)
            ht = float(ht)

            bmi = round(wt / ((ht / 100)**2),0)
    else:
        bmi = None

    all_bmi.append(bmi)

In [29]:
lab_df['Body Mass Index (BMI)'] = all_bmi

In [30]:
lab_df.head()

,sample.sampleKitGuid,Body Mass Index (BMI),Height,Weight,Alanine Transaminase (ALT),Albumin,Alkaline Phosphatase,Aspartate Aminotransferase (AST),"Bilirubin, Total (T-Bili)",Blood Urea Nitrogen (BUN),...,RFIgA Result,RFIgM Interpretation,RFIgM Result,SED Rate-Westergren (ESR),"Cholesterol, HDL","Cholesterol, LDL","Cholesterol, Non-HDL","Cholesterol, Total",Cholesterol/HDL Ratio,Triglycerides
0,KT00001,23.0,177.4,71.1,8,4.7,47,16,0.5,11,...,0,Negative,5.902,2,65,85,104,169,2.6,94
1,KT00002,22.0,180.3,72.6,16,4.5,35,19,0.4,19,...,None,None,None,2,56,72,99,155,2.8,197
2,KT00003,21.0,175.3,64.4,11,4.5,45,16,1.7,10,...,None,None,None,2,76,112,126,202,2.7,57
3,KT00004,22.0,180.3,69.9,15,4.2,45,18,0.5,11,...,None,None,None,2,60,63,76,136,2.3,47
4,KT00006,20.0,170,59.2,12,4.5,35,13,0.3,8,...,None,None,None,2,50,79,102,152,3,130


In [31]:
lab_df = pl.from_pandas(lab_df)

In [32]:
lab_df = lab_df.rename(lab_names)

In [33]:
lab_df.head()

sample.sampleKitGuid,am.bmi,am.height,am.weight,chem.alt,chem.albumin,chem.alkaline_phosphatase,chem.ast,chem.t_bili,chem.bun,chem.calcium,chem.co2,chem.cl,chem.creatinine,chem.egfr_aa,chem.egfr_non_aa,chem.globin,chem.glucose,chem.ldh,chem.magnesium,chem.phosphate,chem.potassium,chem.protein,chem.sodium,bc.perc_basophils,bc.perc_eosinophils,bc.perc_lymphocytes,bc.perc_monocytes,bc.perc_neutrophils,bc.basophil_count,bc.eosinophil_count,bc.lymphocyte_count,bc.monocyte_count,bc.neutrophil_count,bc.hematocrit,bc.hemoglobin,bc.mch,bc.mchc,bc.mcv,bc.mpv,bc.platelet_count,bc.red_blood_cell_count,bc.rdw,bc.wbc,cmv.igg_serology,cmv.igg_serology_interpretation,infl.anti_ccp3,infl.anti_ccp31,infl.hs_crp,infl.rf_iga_interpretation,infl.rf_iga_result,infl.rf_igm_interpretation,infl.rf_igm_result,infl.esr,lip.cholesterol_hdl,lip.cholesterol_ldl,lip.cholesterol_non_hdl,lip.cholesterol_total,lip.chlesterol_hdl_ratio,lip.triglycerides
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,null,null,null,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,f64,str,f64,str,f64,i64,i64,i64,i64,i64,str,i64
"""KT00001""",23.0,177.4,71.1,8.0,4.7,47.0,16.0,0.5,11.0,9.5,22.0,106.0,0.93,94,81,2.6,33.0,null,null,null,3.9,7.0,143.0,0.5,1.6,31.1,7.7,59.1,22.0,69.0,1337.0,331.0,2541.0,33.6,11.0,26.7,32.4,82.2,12.8,226.0,4.0,15.1,4.3,null,null,2,4,0.3,"""Negative""",0.0,"""Negative""",5.902,2,65,85,104,169,"""2.6""",94
"""KT00002""",22.0,180.3,72.6,16.0,4.5,35.0,19.0,0.4,19.0,9.2,25.0,101.0,0.94,127,110,2.3,54.0,null,null,null,3.3,7.0,140.0,0.9,2.4,39.5,8.9,48.3,50.0,132.0,2173.0,490.0,2657.0,42.4,14.0,31.5,34.0,92.8,10.2,224.0,5.0,11.9,5.5,null,null,null,null,0.3,null,null,null,null,2,56,72,99,155,"""2.8""",197
"""KT00003""",21.0,175.3,64.4,11.0,4.5,45.0,16.0,1.7,10.0,9.2,26.0,103.0,0.67,137,118,2.8,47.0,null,null,null,3.9,7.0,144.0,0.7,0.7,45.4,12.1,41.1,29.0,29.0,1861.0,496.0,1685.0,37.6,13.0,30.0,34.0,88.3,11.8,203.0,4.0,12.1,4.1,null,null,null,null,1.9,null,null,null,null,2,76,112,126,202,"""2.7""",57
"""KT00004""",22.0,180.3,69.9,15.0,4.2,45.0,18.0,0.5,11.0,8.9,23.0,106.0,0.88,134,115,2.7,78.0,null,null,null,3.9,7.0,141.0,1.1,2.5,49.8,10.9,35.7,32.0,73.0,1444.0,316.0,1035.0,37.3,13.0,30.8,33.8,91.2,9.8,216.0,4.0,12.2,2.9,null,null,null,null,0.3,null,null,null,null,2,60,63,76,136,"""2.3""",47
"""KT00006""",20.0,170.0,59.2,12.0,4.5,35.0,13.0,0.3,8.0,9.2,26.0,102.0,0.63,142,123,2.4,44.0,null,null,null,3.7,7.0,140.0,0.7,0.1,19.0,11.3,68.9,52.0,7.0,1406.0,836.0,5099.0,39.2,13.0,30.3,33.4,90.5,10.0,254.0,4.0,12.0,7.4,null,null,null,null,0.5,null,null,null,null,2,50,79,102,152,"""3""",130


### Combine with sample metadata

In [34]:
combined_df = meta.join(lab_df, how = 'left', on = 'sample.sampleKitGuid')

In [35]:
combined_df.shape

(108, 74)

In [36]:
out_file = 'output/human_immune_health_atlas_metadata_labs_{d}.csv'.format(d = date.today())
combined_df.write_csv(out_file)

## Upload data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [37]:
study_space_uuid = '64097865-486d-43b3-8f94-74994e0a72e0'
title = 'Clinical Labs and Metadata for File Set {d}'.format(d = date.today())

In [38]:
search_id = element_id()
search_id

'erbium-nickel-arsenic'

In [39]:
in_files = [grouping_uuid, meta_uuid]
in_files

['0180d34e-bcb4-4b81-9214-818641ec54e3',
 '2da66a1a-17cc-498b-9129-6858cf639caf']

In [40]:
out_files = [out_file]

In [41]:
out_files

['output/human_immune_health_atlas_metadata_labs_2025-06-09.csv']

In [42]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

checking if conda environment can compile...
Cannot determine the current notebook.
1) /home/workspace/aifi-healthy-pbmc-reference/05-Assembly/29-Python_clean_all_genes_normalized.ipynb
2) /home/workspace/aifi-healthy-pbmc-reference/09-Clinical_labs/37a-Python_metadata_and_labs_fileset.ipynb
3) /home/workspace/temp/data-apps-vis/plotly/ra_infl_marker_distribution_long.ipynb
Please select (1-3) 


 2


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '0bd0f22e-4157-49f3-96ed-7b321b4a6816',
 'ProcessId': 'e68c2427-b37d-4d1f-b585-159373c91e77',
 'WorkflowId': '158ac0f9-f3de-41cb-958c-9ecdce7e7ab9',
 'FileIds': ['cffc47a7-4f52-4a28-b0e8-c9b219ef3741']}

In [44]:
import session_info
session_info.show()